# Modesty Blur: train the person classifier

Run the cells top to bottom (Runtime > Run all). First turn on a GPU: **Runtime > Change runtime type > T4 GPU**.

Your pictures stay in your own Google Drive and in this private Colab session. Nothing is uploaded anywhere else.

The Drive folder must look like this (the app's folder is already set up for you):
```
ModestyBlur-dataset/
  woman/  man/  girl/  boy/  teen_girl/  teen_boy/      <- pictures, or .zip files of pictures, in each
```
Zip files (1-2 GB each) load MUCH faster than hundreds of thousands of loose files.

In [ ]:
#@title 1. Connect Google Drive (optional: skip with Cancel if your pictures are uploaded directly)
import os
DRIVE_DIR = '/content/drive/MyDrive/ModestyBlur-dataset'   #@param {type:"string"}
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print(os.listdir(DRIVE_DIR))
except Exception as e:
    print('Drive not available (that is fine if you uploaded the zips to Colab):', e)

In [ ]:
#@title 2. Get the code and tools
!rm -rf /content/repo
!git clone -q --depth 1 https://github.com/Nex-Gen-Dev/Modesty-blur /content/repo
!pip -q install onnx onnxruntime pillow
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable a GPU runtime!')

In [ ]:
#@title 3. Copy the pictures to this machine's fast disk (unzips the .zip files)
#@markdown Where the zips come from, in this order: the Drive folder, or zip files you dragged into Colab's file panel
#@markdown (left side, folder icon) named woman.zip, man.zip, girl.zip, boy.zip, teen_girl.zip, teen_boy.zip.
import os, shutil, subprocess, glob
CLASSES = ['woman','man','girl','boy','teen_girl','teen_boy']
LOCAL = '/content/data'
for c in CLASSES:
    dst = os.path.join(LOCAL, c)
    os.makedirs(dst, exist_ok=True)
    src = os.path.join(DRIVE_DIR, c)
    zips = sorted(glob.glob(os.path.join(src, '*.zip'))) + sorted(glob.glob(f'/content/{c}.zip')) + sorted(glob.glob(f'/content/upload/{c}.zip'))
    for z in zips:
        print(c, ': unzipping', z)
        sub = os.path.join(dst, os.path.splitext(os.path.basename(z))[0] + '_' + str(abs(hash(z)) % 10000))   # one folder per zip, so names never clash
        subprocess.run(['unzip', '-q', '-o', z, '-d', sub], check=False)
    if os.path.isdir(src):
        subprocess.run(['rsync', '-a', '--exclude=*.zip', src + '/', dst + '/'], check=False)
    n = sum(len(f) for _, _, f in os.walk(dst))
    print(f'{c:10s}: {n} files', '   <-- NONE FOUND, check the file names' if n == 0 else '')

In [ ]:
#@title 4. (Only if your pictures are WHOLE SCENES with many people) cut people out
#@markdown Leave unchecked if each picture already shows one person.
CUT_OUT_PEOPLE = False  #@param {type:"boolean"}
if CUT_OUT_PEOPLE:
    !pip -q install ultralytics
    for c in CLASSES:
        !python /content/repo/tools/extract_crops.py /content/data/{c} --out /content/crops/{c} --every 1
    LOCAL = '/content/crops'
print('training from', LOCAL)

In [ ]:
#@title 5. Train (about 10-40 minutes on a T4 for 200,000 pictures)
EPOCHS = 12          #@param {type:"integer"}
FEMALE_WEIGHT = 3.0  #@param {type:"number"}
#@markdown FEMALE_WEIGHT: how much worse it is to miss a woman/girl than to blur a man/boy. Higher = fewer misses, more false blurs.
!cd /content/repo && python tools/train_person_cls.py {LOCAL} --out /content/out --epochs {EPOCHS} --female-weight {FEMALE_WEIGHT} --workers 2 --batch 128 --shrink /content/small

In [ ]:
#@title 6. Save the result
import shutil, os
try:
    os.makedirs(DRIVE_DIR + '/_output', exist_ok=True)
    for f in ['person_cls.onnx', 'person_cls.json']:
        shutil.copy('/content/out/' + f, DRIVE_DIR + '/_output/' + f)
    print('Saved to Drive:', DRIVE_DIR + '/_output/')
except Exception as e:
    print('Could not save to Drive (the download below still works):', e)
from google.colab import files
files.download('/content/out/person_cls.onnx'); files.download('/content/out/person_cls.json')
print('Send those two files to Claude (attach them in the chat). They are small (about 6 MB).')